<a href="https://colab.research.google.com/github/Sathvika-68/MLops-External_731/blob/main/Mlops_4731.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()

zip_files = [name for name in uploaded if name.lower().endswith(".zip")]
if not zip_files:
    raise ValueError("Please upload shrimp_oxygen_early_warning_complete.zip")
ZIP_NAME = zip_files[0]
print("Uploaded:", ZIP_NAME)


Saving shrimp_oxygen_early_warning_complete (5).zip to shrimp_oxygen_early_warning_complete (5).zip
Uploaded: shrimp_oxygen_early_warning_complete (5).zip


## Cell 2 — Extract the project and enter its folder

In [ ]:
import os
import zipfile
import shutil
from pathlib import Path

EXTRACT_ROOT = Path("/content/shrimp_project_extract")
PROJECT_DIR = Path("/content/shrimp_oxygen_early_warning")

if EXTRACT_ROOT.exists():
    shutil.rmtree(EXTRACT_ROOT)
if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_NAME, "r") as zf:
    zf.extractall(EXTRACT_ROOT)

# The supplied ZIP contains the project directory as its top-level folder.
project_candidates = [
    p for p in EXTRACT_ROOT.iterdir()
    if p.is_dir() and (p / "requirements.txt").exists()
]
if project_candidates:
    shutil.copytree(project_candidates[0], PROJECT_DIR)
elif (EXTRACT_ROOT / "requirements.txt").exists():
    shutil.copytree(EXTRACT_ROOT, PROJECT_DIR)
else:
    raise FileNotFoundError("Could not find requirements.txt in the extracted ZIP.")

os.chdir(PROJECT_DIR)
print("Project folder:", os.getcwd())
print("Project files:", sorted(p.name for p in PROJECT_DIR.iterdir()))


Project folder: /content/shrimp_oxygen_early_warning
Project files: ['.github', '.gitignore', 'COLAB_INSTRUCTIONS.md', 'Dockerfile', 'PROJECT_CONTENTS.txt', 'README.md', 'data', 'reports', 'requirements.txt', 'src', 'tests']


## Cell 3 — Install the project dependencies

In [ ]:
%pip install -q --upgrade pip
%pip install -q -r requirements.txt
print("Dependencies installed.")


ERROR: Invalid requirement: 'pandas>=2.0\\nnumpy>=1.24\\nscikit-learn>=1.3\\njoblib>=1.3\\nmlflow>=3.10\\nfastapi>=0.110\\nuvicorn[standard]>=0.27\\npydantic>=2.0\\npytest>=8.0\\nhttpx>=0.27\\n': Expected comma (within version specifier), semicolon (after version specifier) or end
    pandas>=2.0\nnumpy>=1.24\nscikit-learn>=1.3\njoblib>=1.3\nmlflow>=3.10\nfastapi>=0.110\nuvicorn[standard]>=0.27\npydantic>=2.0\npytest>=8.0\nhttpx>=0.27\n
          ~~~~~^ (from line 1 of requirements.txt)
Dependencies installed.


## Cell 4 — Generate V1, V2 and reusable feature files

In [ ]:
import pandas as pd
from src.data_pipeline import create_versions, build_feature_store

v1, v2 = create_versions(n=300)
v1_features = build_feature_store("v1")
v2_features = build_feature_store("v2")

print("V1 rows:", len(v1))
print("V1 target distribution:", v1["oxygen_risk"].value_counts().to_dict())
print("V2 rows:", len(v2))
print("V2 target distribution:", v2["oxygen_risk"].value_counts().to_dict())
print("Feature files:", v1_features, v2_features)

display(v1.head())
display(pd.read_csv(v1_features).head())


V1 rows: 300
V1 target distribution: {'Normal': 192, 'High': 108}
V2 rows: 300
V2 target distribution: {'High': 190, 'Normal': 110}
Feature files: /content/shrimp_oxygen_early_warning/data/features/features_v1.csv /content/shrimp_oxygen_early_warning/data/features/features_v2.csv


,pond_id,pond_area_ha,stocking_density_per_m2,do_8pm_mg_l,water_temp_c,daily_feed_kg,planned_aerator_hours,expected_rainfall_mm,oxygen_risk
0,P01,2.435,84.53,5.76,28.26,1.06,3.86,7.68,High
1,P02,1.597,39.42,6.17,28.35,0.52,1.61,2.78,Normal
2,P03,2.646,67.52,6.41,26.93,1.04,2.56,1.61,Normal
3,P04,2.243,66.00,5.44,25.57,0.84,4.09,1.87,Normal
4,P05,0.735,90.03,6.74,28.79,0.56,6.20,1.57,Normal


,pond_id,pond_area_ha,stocking_density_per_m2,do_8pm_mg_l,water_temp_c,daily_feed_kg,planned_aerator_hours,expected_rainfall_mm,oxygen_risk,feed_per_unit_biomass,aeration_hours_per_area
0,P01,2.435,84.53,5.76,28.26,1.06,3.86,7.68,High,0.025749,1.585216
1,P02,1.597,39.42,6.17,28.35,0.52,1.61,2.78,Normal,0.041300,1.008140
2,P03,2.646,67.52,6.41,26.93,1.04,2.56,1.61,Normal,0.029106,0.967498
3,P04,2.243,66.00,5.44,25.57,0.84,4.09,1.87,Normal,0.028371,1.823451
4,P05,0.735,90.03,6.74,28.79,0.56,6.20,1.57,Normal,0.042314,8.435374


In [ ]:
%pip install -q mlflow

In [ ]:
import mlflow
import mlflow.sklearn

print("MLflow installed successfully!")
print("Version:", mlflow.__version__)

MLflow installed successfully!
Version: 3.18.0


In [ ]:
from pathlib import Path

train_file = Path("/content/shrimp_oxygen_early_warning/src/train.py")

source = train_file.read_text()

source = source.replace(
    'mlflow.sklearn.log_model(pipe, artifact_path="model")',
    'mlflow.sklearn.log_model(pipe, artifact_path="model", serialization_format="cloudpickle")'
)

train_file.write_text(source)

print("MLflow serialization compatibility fixed!")

MLflow serialization compatibility fixed!


## Cell 5 — Train Logistic Regression and Random Forest on V1 and V2
This runs the complete training pipeline. MLflow tracking is configured to use `sqlite:///mlflow.db`, not the deprecated filesystem tracking backend.

In [ ]:
import importlib
import src.train as train_module

importlib.reload(train_module)

print("Training module reloaded successfully!")
print("========== TRAINING V1 ==========")

selected_v1, results_v1 = train_module.train("v1")

print("\n========== TRAINING V2 ==========")

selected_v2, results_v2 = train_module.train("v2")

print("\nSelected V1:", selected_v1)
print("Selected V2:", selected_v2)

2026/11/18 09:47:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/11/18 09:47:26 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Training module reloaded successfully!
========== TRAINING V1 ==========


2026/11/18 09:47:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/11/18 09:47:32 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'ShrimpOxygenRiskModel'.
2026/11/18 09:47:36 WARNING mlflow.tracking._model_registry.fluent: Run with id df27a13dcca04fb38f8898d505fd5118 has no artifacts at artifact path 'model', registering model based on models:/m-3cecca18410743c08351a402e8c47698 instead
Created version '1' of model 'ShrimpOxygenRiskModel'.
2026/11/18 09:47:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/11/18 09:47:36 WARNING mlflow.sklearn

{
  "selected_model": "LogisticRegression",
  "metrics": [
    {
      "model": "LogisticRegression",
      "accuracy": 0.8533333333333334,
      "high_risk_precision": 0.8076923076923077,
      "high_risk_recall": 0.7777777777777778,
      "high_risk_f1": 0.7924528301886793,
      "run_id": "df27a13dcca04fb38f8898d505fd5118"
    },
    {
      "model": "RandomForest",
      "accuracy": 0.7733333333333333,
      "high_risk_precision": 0.7083333333333334,
      "high_risk_recall": 0.6296296296296297,
      "high_risk_f1": 0.6666666666666666,
      "run_id": "977f7b92df3c46dca6f38ce858154318"
    }
  ]
}

========== TRAINING V2 ==========


2026/11/18 09:47:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/11/18 09:47:40 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'ShrimpOxygenRiskModel' already exists. Creating a new version of this model...
2026/11/18 09:47:43 WARNING mlflow.tracking._model_registry.fluent: Run with id 62ab32ac330046889b5653391d1a4aec has no artifacts at artifact path 'model', registering model based on models:/m-6efda6e69f784368907b2ea9a5f1eee3 instead


{
  "selected_model": "RandomForest",
  "metrics": [
    {
      "model": "LogisticRegression",
      "accuracy": 0.92,
      "high_risk_precision": 0.9767441860465116,
      "high_risk_recall": 0.8936170212765957,
      "high_risk_f1": 0.9333333333333333,
      "run_id": "0d48b9aa054a420585374cf488af73ec"
    },
    {
      "model": "RandomForest",
      "accuracy": 0.7866666666666666,
      "high_risk_precision": 0.7818181818181819,
      "high_risk_recall": 0.9148936170212766,
      "high_risk_f1": 0.8431372549019608,
      "run_id": "62ab32ac330046889b5653391d1a4aec"
    }
  ]
}

Selected V1: LogisticRegression
Selected V2: RandomForest


Created version '2' of model 'ShrimpOxygenRiskModel'.


In [ ]:
import mlflow

mlflow.set_tracking_uri("sqlite:///mlflow.db")

print("MLflow version:", mlflow.__version__)
print("Tracking URI:", mlflow.get_tracking_uri())


MLflow version: 3.18.0
Tracking URI: sqlite:///mlflow.db


## Cell 6 — Display model comparison metrics

In [ ]:
import json
from IPython.display import display, Markdown

for version in ["v1", "v2"]:
    report_path = Path("reports") / f"model_comparison_{version}.json"
    report = json.loads(report_path.read_text())
    print(f"\\n{version.upper()} — selected model: {report['selected_model']}")
    display(pd.DataFrame(report["metrics"])[
        ["model", "accuracy", "high_risk_precision", "high_risk_recall", "high_risk_f1"]
    ])


\nV1 — selected model: LogisticRegression


,model,accuracy,high_risk_precision,high_risk_recall,high_risk_f1
0,LogisticRegression,0.853333,0.807692,0.777778,0.792453
1,RandomForest,0.773333,0.708333,0.629630,0.666667


\nV2 — selected model: RandomForest


,model,accuracy,high_risk_precision,high_risk_recall,high_risk_f1
0,LogisticRegression,0.920000,0.976744,0.893617,0.933333
1,RandomForest,0.786667,0.781818,0.914894,0.843137


## Cell 7 — Verify MLflow tracking and model registry

In [ ]:
import mlflow
from mlflow.tracking import MlflowClient

mlflow.set_tracking_uri("sqlite:///mlflow.db")
client = MlflowClient()

experiment = client.get_experiment_by_name("shrimp-oxygen-early-warning")
print("Experiment:", experiment.name if experiment else "Not found")

if experiment:
    runs = client.search_runs(
        experiment_ids=[experiment.experiment_id],
        order_by=["attributes.start_time DESC"]
    )
    run_rows = []
    for run in runs:
        run_rows.append({
            "run_name": run.data.tags.get("mlflow.runName", ""),
            "model": run.data.params.get("model", ""),
            "dataset_version": run.data.params.get("dataset_version", ""),
            "high_risk_precision": run.data.metrics.get("high_risk_precision"),
            "high_risk_recall": run.data.metrics.get("high_risk_recall"),
            "high_risk_f1": run.data.metrics.get("high_risk_f1"),
        })
    display(pd.DataFrame(run_rows))

try:
    registered = client.search_registered_models()
    print("Registered models:", [m.name for m in registered])
except Exception as exc:
    print("Registry lookup message:", exc)


Experiment: shrimp-oxygen-early-warning


,run_name,model,dataset_version,high_risk_precision,high_risk_recall,high_risk_f1
0,v2-RandomForest,RandomForest,v2,0.781818,0.914894,0.843137
1,v2-LogisticRegression,LogisticRegression,v2,0.976744,0.893617,0.933333
2,v1-RandomForest,RandomForest,v1,0.708333,0.629630,0.666667
3,v1-LogisticRegression,LogisticRegression,v1,0.807692,0.777778,0.792453
4,v1-LogisticRegression,LogisticRegression,v1,0.807692,0.777778,0.792453


Registered models: ['ShrimpOxygenRiskModel']


## Cell 8 — Run automated tests

In [ ]:
!python -m pytest -q


...                                                                      [100%]
3 passed in 0.95s


## Cell 9 — Test FastAPI `/health` and `/predict` in Colab
This uses FastAPI TestClient. It tests the API without needing to expose a public server.

In [ ]:
from fastapi.testclient import TestClient
from src.api import app

client = TestClient(app)

health_response = client.get("/health")
print("GET /health:", health_response.status_code, health_response.json())

sample_pond = {
    "pond_area_ha": 1.2,
    "stocking_density_per_m2": 85,
    "do_8pm_mg_l": 5.1,
    "water_temp_c": 30.5,
    "daily_feed_kg": 420,
    "planned_aerator_hours": 3,
    "expected_rainfall_mm": 12
}

prediction_response = client.post("/predict", json=sample_pond)
print("POST /predict:", prediction_response.status_code)
print(json.dumps(prediction_response.json(), indent=2))


GET /health: 200 {'status': 'ok', 'model_loaded': True}
POST /predict: 200
{
  "oxygen_risk": "High",
  "high_risk_probability": 0.9621,
  "additional_night_aeration_recommended": true,
  "recommendation": "Consider increasing night-time aeration and verify DO with a calibrated meter."
}


## Cell 10 — Optional: start the MLflow tracking server
This starts MLflow in the Colab runtime. The SQLite database and model artifacts are saved in the project folder. If Colab does not expose the port in your session, you can still inspect all runs in Cell 7.

In [ ]:
import subprocess
import time

mlflow_log_path = "/content/mlflow_server.log"
mlflow_log = open(mlflow_log_path, "w")
mlflow_process = subprocess.Popen(
    [
        "python", "-m", "mlflow", "server",
        "--backend-store-uri", "sqlite:///mlflow.db",
        "--host", "0.0.0.0",
        "--port", "5000"
    ],
    cwd=str(PROJECT_DIR),
    stdout=mlflow_log,
    stderr=subprocess.STDOUT
)
time.sleep(3)
print("MLflow server process:", mlflow_process.poll() is None)
print("Server log:", mlflow_log_path)
print("Tracking database:", PROJECT_DIR / "mlflow.db")


MLflow server process: True
Server log: /content/mlflow_server.log
Tracking database: /content/shrimp_oxygen_early_warning/mlflow.db


## Cell 11 — Download the completed project and outputs
This creates a ZIP containing the source code, generated datasets, engineered features, trained model, comparison reports, SQLite MLflow tracking database and PDF report.

In [ ]:
from google.colab import files
import shutil

os.chdir("/content")
OUTPUT_ZIP = shutil.make_archive(
    "/content/shrimp_oxygen_early_warning_colab_results",
    "zip",
    root_dir="/content",
    base_dir="shrimp_oxygen_early_warning"
)
print("Created:", OUTPUT_ZIP)
files.download(OUTPUT_ZIP)


Created: /content/shrimp_oxygen_early_warning_colab_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Optional — Run the API as a server

The notebook already tested both API endpoints using TestClient. If you want to start the web server inside the Colab runtime, run:

```python
%cd /content/shrimp_oxygen_early_warning
!uvicorn src.api:app --host 0.0.0.0 --port 8000
```

The server runs inside the Colab runtime; this command alone does not create a permanent public deployment URL. For a public endpoint, deploy the project to a hosting platform and configure HTTPS, authentication and monitoring.

## What is included in the project?
- `src/data_pipeline.py` — synthetic data, validation and feature engineering.
- `src/train.py` — Logistic Regression, Random Forest, metrics and MLflow registry.
- `src/predict.py` — prediction helper.
- `src/api.py` — FastAPI endpoints.
- `data/raw/` — V1 and V2 datasets.
- `data/features/` — reusable engineered feature CSV files.
- `tests/` — data and prediction tests.
- `.github/workflows/ci.yml` — GitHub Actions workflow.
- `reports/` — PDF report and model comparison JSON files.
